# Notebook 04: Model Evaluation & Residual Diagnostics

## CRISP-DM Phase: Evaluation

**Project**: OptiWMS — AI-Driven Warehouse Management System  
**Reference**: Petropoulos et al. (2022), Section 2.12 (Forecast Evaluation)

---

### Evaluation Strategy

This notebook goes beyond simple metric comparison to perform **diagnostic-level evaluation** as required for academic rigour:

1. **R-squared analysis** — explained variance per model
2. **Residual analysis** — QQ plots, ACF of residuals, normality tests
3. **Statistical tests** — Shapiro-Wilk (normality), Ljung-Box (autocorrelation), Diebold-Mariano (forecast comparison)
4. **Scale-free metrics** — MASE (Hyndman & Koehler, 2006)
5. **Probabilistic evaluation** — Pinball loss for quantile forecasts
6. **Bias-variance analysis** — over/under-forecasting by segment

> *"The evaluation of forecast accuracy should be carried out using multiple complementary metrics."*  
> — Petropoulos et al. (2022), Section 2.12.1

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
SEED = 42

ROOT = Path('..').resolve()
DATA_DIR = ROOT.parent / 'Forecast model train data optiwms'
GEN_DIR  = ROOT / 'outputs' / 'generated'
ENG_DIR  = ROOT / 'outputs' / 'engineered'

print('Libraries loaded')

In [ ]:
# Load data and retrain models (or load from NB03 artifacts)
def _aggregate_fg_monthly(fg):
    fg = fg.copy()
    fg['month'] = pd.to_datetime(fg['month']).dt.to_period('M').dt.to_timestamp()
    if 'demand_units_clean' in fg.columns and 'demand_units' not in fg.columns:
        fg['demand_units'] = fg['demand_units_clean']
    num_cols = fg.select_dtypes(include=[np.number]).columns.tolist()
    agg = {c: 'mean' for c in num_cols if c != 'demand_units'}
    if 'demand_units' in fg.columns:
        agg['demand_units'] = 'mean'  # 60 MC scenarios/SKU-month — mean = expected demand (sum would 60x inflate)
    for c in [c for c in fg.columns if c not in agg and c not in ('fg_code', 'month')]:
        agg[c] = 'first'
    return fg.groupby(['fg_code', 'month'], as_index=False).agg(agg).sort_values(['fg_code', 'month']).reset_index(drop=True)

if (ENG_DIR / 'fg_features_engineered.csv').exists():
    fg = pd.read_csv(ENG_DIR / 'fg_features_engineered.csv')
    fg['month'] = pd.to_datetime(fg['month'])
    if len(fg) > 20_000:
        print('Re-aggregating bloated engineered file to monthly panel...')
        fg = _aggregate_fg_monthly(fg)
else:
    fg = pd.read_csv(DATA_DIR / 'hemas_scenario_c_dataset_cleaned.csv')
    fg = _aggregate_fg_monthly(fg)

print(f'FG panel: {len(fg):,} rows')

# Split
sorted_months = sorted(fg['month'].unique())
n_months = len(sorted_months)
train_end = sorted_months[min(23, n_months-7)]
val_end = sorted_months[min(29, n_months-1)]

train_df = fg[fg['month'] <= train_end].copy()
val_df   = fg[(fg['month'] > train_end) & (fg['month'] <= val_end)].copy()
test_df  = fg[fg['month'] > val_end].copy()

feature_cols = [
    'month_num', 'quarter', 'year', 'is_year_end', 'is_sl_peak',
    'month_sin', 'month_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'demand_lag_6', 'demand_lag_12',
    'demand_rmean_3', 'demand_rmean_6', 'demand_rstd_3', 'demand_rstd_6',
    'demand_rmin_3', 'demand_rmax_3', 'demand_rmin_6', 'demand_rmax_6',
    'demand_cv_6', 'demand_momentum',
    'fg_category_enc', 'fg_code_enc',
]
for col in ['on_hand_inventory', 'lead_time_days', 'supplier_otif',
            'promotion_flag', 'holiday_flag', 'price_per_unit']:
    if col in fg.columns:
        feature_cols.append(col)

available_feats = [c for c in feature_cols if c in fg.columns]
TARGET = 'demand_units'

X_train = train_df[available_feats].values
y_train = train_df[TARGET].values
X_val = val_df[available_feats].values
y_val = val_df[TARGET].values

print(f'Train: {X_train.shape}, Val: {X_val.shape}')


In [ ]:
# Retrain best models quickly
lgb_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                               num_leaves=63, verbose=-1, random_state=SEED)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(30, verbose=False)])

cat_model = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=8,
                               random_seed=SEED, verbose=0, early_stopping_rounds=30)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=0)

xgb_model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                              random_state=SEED, verbosity=0, early_stopping_rounds=30)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# Generate predictions
preds = {
    'LightGBM': np.clip(lgb_model.predict(X_val), 0, None),
    'CatBoost': np.clip(cat_model.predict(X_val), 0, None),
    'XGBoost':  np.clip(xgb_model.predict(X_val), 0, None),
}

# Seasonal naive
snaive_pred = []
for _, row in val_df.iterrows():
    same_m = train_df[(train_df['fg_code'] == row['fg_code']) &
                       (train_df['month'].dt.month == row['month'].month)]
    snaive_pred.append(same_m['demand_units'].iloc[-1] if len(same_m) > 0 else train_df[train_df['fg_code'] == row['fg_code']]['demand_units'].mean())
preds['Seasonal Naive'] = np.array(snaive_pred)

print('Models trained and predictions generated')

## 1. R-Squared Analysis

R-squared measures the proportion of variance in actual demand explained by the model.  
R^2 = 1 means perfect prediction; R^2 = 0 means no better than predicting the mean.

In [ ]:
# 1.1 Overall R-squared
r2_scores = {name: r2_score(y_val, pred) for name, pred in preds.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = ['#3498db' if r > 0.5 else '#e74c3c' for r in r2_scores.values()]
axes[0].bar(r2_scores.keys(), r2_scores.values(), color=colors)
axes[0].axhline(0.5, color='orange', linestyle='--', label='Good threshold (0.5)')
axes[0].axhline(0.8, color='green', linestyle='--', label='Excellent threshold (0.8)')
axes[0].set_ylabel('R-Squared')
axes[0].set_title('R-Squared by Model')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)
for i, (name, r2) in enumerate(r2_scores.items()):
    axes[0].text(i, r2 + 0.02, f'{r2:.3f}', ha='center', fontweight='bold')

# Actual vs Predicted scatter (best model)
best_name = max(r2_scores, key=r2_scores.get)
best_pred = preds[best_name]
axes[1].scatter(y_val, best_pred, alpha=0.1, s=5)
lim = max(y_val.max(), best_pred.max())
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Demand')
axes[1].set_ylabel('Predicted Demand')
axes[1].set_title(f'Actual vs Predicted — {best_name} (R²={r2_scores[best_name]:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

print('=== R-Squared Summary ===')
for name, r2 in sorted(r2_scores.items(), key=lambda x: -x[1]):
    quality = 'Excellent' if r2 > 0.8 else 'Good' if r2 > 0.5 else 'Poor'
    print(f'  {name}: R² = {r2:.4f} ({quality})')

## 2. Residual Analysis

> *"Residual diagnostics verify that the model has captured all systematic patterns in the data."*  
> — Petropoulos et al. (2022), Section 2.12.3

For a good model, residuals should be:
1. **Zero-centred** (no systematic bias)
2. **Approximately normal** (QQ plot / Shapiro-Wilk)
3. **No autocorrelation** (ACF / Ljung-Box test)
4. **Constant variance** (homoscedasticity)

In [ ]:
# 2.1 Residual analysis for best ML model
residuals = y_val - best_pred

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (a) Residual histogram + KDE
sns.histplot(residuals, bins=60, kde=True, ax=axes[0, 0])
axes[0, 0].axvline(0, color='red', linestyle='--')
axes[0, 0].set_title(f'Residual Distribution — {best_name}')
axes[0, 0].set_xlabel('Residual (actual - predicted)')

# (b) QQ Plot
stats.probplot(residuals, plot=axes[0, 1])
axes[0, 1].set_title('QQ Plot (Normality Check)')
axes[0, 1].get_lines()[0].set_markersize(2)

# (c) Residuals vs Predicted (homoscedasticity check)
axes[0, 2].scatter(best_pred, residuals, alpha=0.1, s=5)
axes[0, 2].axhline(0, color='red', linestyle='--')
axes[0, 2].set_xlabel('Predicted')
axes[0, 2].set_ylabel('Residual')
axes[0, 2].set_title('Residuals vs Predicted (Homoscedasticity)')

# (d) ACF of residuals
plot_acf(residuals[:min(500, len(residuals))], lags=20, ax=axes[1, 0],
         title='ACF of Residuals')

# (e) Residuals over time
axes[1, 1].scatter(range(len(residuals)), residuals, alpha=0.1, s=3)
axes[1, 1].axhline(0, color='red', linestyle='--')
axes[1, 1].set_xlabel('Observation Index')
axes[1, 1].set_ylabel('Residual')
axes[1, 1].set_title('Residuals Over Time')

# (f) Standardised residual box by category
val_df_copy = val_df.copy()
val_df_copy['residual'] = residuals
top_cats = val_df_copy['fg_category'].value_counts().head(5).index
sns.boxplot(data=val_df_copy[val_df_copy['fg_category'].isin(top_cats)],
            x='fg_category', y='residual', ax=axes[1, 2])
axes[1, 2].axhline(0, color='red', linestyle='--')
axes[1, 2].set_title('Residuals by Category')
axes[1, 2].tick_params(axis='x', rotation=30)

plt.suptitle(f'Residual Diagnostics — {best_name}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Statistical Tests

In [ ]:
# 3.1 Shapiro-Wilk test for normality of residuals
# (use sample if > 5000 observations)
resid_sample = residuals[:5000] if len(residuals) > 5000 else residuals
shapiro_stat, shapiro_p = stats.shapiro(resid_sample)

print('=== Shapiro-Wilk Normality Test ===')
print(f'  Statistic: {shapiro_stat:.4f}')
print(f'  p-value: {shapiro_p:.6f}')
print(f'  Conclusion: Residuals are {"approximately normal" if shapiro_p > 0.05 else "NOT normal"} (alpha=0.05)')
print(f'  Note: For large samples, even slight deviations cause rejection. Visual QQ plot is more informative.')

# 3.2 Ljung-Box test for autocorrelation in residuals
lb_result = acorr_ljungbox(residuals[:min(1000, len(residuals))], lags=[6, 12], return_df=True)

print('\n=== Ljung-Box Autocorrelation Test ===')
print(lb_result)
print(f'  Conclusion: {"No significant autocorrelation" if lb_result["lb_pvalue"].min() > 0.05 else "Significant autocorrelation detected"}')

# 3.3 Additional stats
print(f'\n=== Residual Summary Statistics ===')
print(f'  Mean: {residuals.mean():.2f} (should be near 0)')
print(f'  Std: {residuals.std():.2f}')
print(f'  Skewness: {stats.skew(residuals):.3f}')
print(f'  Kurtosis: {stats.kurtosis(residuals):.3f}')

## 4. Diebold-Mariano Test (Model Comparison)

> *"The Diebold-Mariano test formally tests whether two forecasts have equal predictive accuracy."*  
> — Petropoulos et al. (2022), Section 2.12.4

H0: Both models have equal forecast accuracy  
H1: One model is significantly better  

We test each ML model against the Seasonal Naive baseline.

In [ ]:
# 4.1 Diebold-Mariano test implementation
def diebold_mariano_test(actual, pred1, pred2, h=1, loss='squared'):
    """Diebold-Mariano test for equal predictive accuracy.
    Returns: DM statistic, p-value
    Negative DM -> pred1 is better; Positive DM -> pred2 is better
    """
    e1 = actual - pred1
    e2 = actual - pred2
    
    if loss == 'squared':
        d = e1**2 - e2**2
    elif loss == 'absolute':
        d = np.abs(e1) - np.abs(e2)
    else:
        d = e1**2 - e2**2
    
    n = len(d)
    d_mean = np.mean(d)
    
    # Autocovariance up to lag h-1
    gamma = [np.mean((d - d_mean) * (d - d_mean))]
    for k in range(1, h):
        gamma.append(np.mean((d[k:] - d_mean) * (d[:-k] - d_mean)))
    
    var_d = (gamma[0] + 2 * sum(gamma[1:])) / n
    if var_d <= 0:
        var_d = 1e-10
    
    dm_stat = d_mean / np.sqrt(var_d)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    
    return dm_stat, p_value

# Run DM tests: each ML model vs Seasonal Naive
print('=== Diebold-Mariano Tests (vs Seasonal Naive) ===')
print(f'{"Model":<20} {"DM Stat":>10} {"p-value":>10} {"Conclusion":<35}')
print('-' * 75)

baseline_pred = preds['Seasonal Naive']
dm_results = []

for name in ['LightGBM', 'CatBoost', 'XGBoost']:
    dm_stat, dm_p = diebold_mariano_test(y_val, preds[name], baseline_pred, h=1)
    if dm_p < 0.05:
        conclusion = f'{name} significantly better' if dm_stat < 0 else 'Seasonal Naive better'
    else:
        conclusion = 'No significant difference'
    print(f'{name:<20} {dm_stat:>10.3f} {dm_p:>10.6f} {conclusion}')
    dm_results.append({'model': name, 'dm_stat': dm_stat, 'p_value': dm_p, 'conclusion': conclusion})

# ML vs ML comparison
print('\n=== Diebold-Mariano: LightGBM vs CatBoost ===')
dm_stat, dm_p = diebold_mariano_test(y_val, preds['LightGBM'], preds['CatBoost'], h=1)
print(f'DM Statistic: {dm_stat:.3f}, p-value: {dm_p:.6f}')
if dm_p < 0.05:
    print(f'Conclusion: {"LightGBM" if dm_stat < 0 else "CatBoost"} is significantly better')
else:
    print('Conclusion: No significant difference between LightGBM and CatBoost')

## 5. MASE — Scale-Free Evaluation

> *"MASE is the recommended scale-free metric for comparing forecasts across series of different scales."*  
> — Hyndman & Koehler (2006); Petropoulos et al. (2022), Section 2.12.2

In [ ]:
# 5.1 Per-SKU MASE computation
def compute_sku_mase(sku, train_data, val_data, predictions, seasonality=12):
    sku_train = train_data[train_data['fg_code'] == sku]['demand_units'].values
    sku_val_idx = val_data[val_data['fg_code'] == sku].index
    
    if len(sku_train) < seasonality + 1 or len(sku_val_idx) == 0:
        return None
    
    naive_errors = np.abs(sku_train[seasonality:] - sku_train[:-seasonality])
    scale = np.mean(naive_errors)
    if scale == 0:
        return None
    
    actual = val_data.loc[sku_val_idx, 'demand_units'].values
    pred_vals = predictions[val_data.index.isin(sku_val_idx)]
    
    return np.mean(np.abs(actual - pred_vals)) / scale

# Compute MASE per SKU per model
mase_per_sku = {}
for name, pred in preds.items():
    mase_vals = []
    for sku in val_df['fg_code'].unique():
        m = compute_sku_mase(sku, train_df, val_df, pred)
        if m is not None:
            mase_vals.append(m)
    mase_per_sku[name] = mase_vals

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MASE distribution
mase_df = pd.DataFrame({name: pd.Series(vals) for name, vals in mase_per_sku.items()})
mase_df.boxplot(ax=axes[0])
axes[0].axhline(1.0, color='red', linestyle='--', label='MASE=1 (Seasonal Naive equivalent)')
axes[0].set_title('MASE Distribution by Model')
axes[0].set_ylabel('MASE')
axes[0].legend()
axes[0].set_ylim(0, min(5, mase_df.quantile(0.95).max()))

# Mean MASE
mean_mase = {name: np.mean(vals) for name, vals in mase_per_sku.items()}
axes[1].bar(mean_mase.keys(), mean_mase.values(),
            color=['#3498db' if v < 1 else '#e74c3c' for v in mean_mase.values()])
axes[1].axhline(1.0, color='red', linestyle='--', label='Baseline (MASE=1)')
axes[1].set_title('Mean MASE by Model (< 1.0 = better than Seasonal Naive)')
axes[1].set_ylabel('Mean MASE')
axes[1].legend()
for i, (name, v) in enumerate(mean_mase.items()):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Pinball Loss — Quantile Forecast Evaluation

> *"Probabilistic forecasts should be evaluated using proper scoring rules such as the pinball loss."*  
> — Petropoulos et al. (2022), Section 2.12.5

In [ ]:
# 6.1 Train quantile models and evaluate
def pinball_loss(y_true, y_pred, tau):
    """Pinball (quantile) loss function."""
    diff = y_true - y_pred
    return np.mean(np.where(diff >= 0, tau * diff, (tau - 1) * diff))

quantiles = {'p10': 0.10, 'p50': 0.50, 'p90': 0.90}
q_predictions = {}

for q_name, alpha in quantiles.items():
    model = lgb.LGBMRegressor(
        objective='quantile', alpha=alpha,
        n_estimators=200, learning_rate=0.05,
        max_depth=8, num_leaves=63,
        verbose=-1, random_state=SEED
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(20, verbose=False)])
    q_predictions[q_name] = np.clip(model.predict(X_val), 0, None)

# Compute pinball loss for each quantile
print('=== Pinball Loss by Quantile ===')
for q_name, alpha in quantiles.items():
    pl = pinball_loss(y_val, q_predictions[q_name], alpha)
    print(f'  {q_name} (tau={alpha}): Pinball Loss = {pl:.2f}')

# Coverage metrics
below_p10 = (y_val < q_predictions['p10']).mean()
above_p90 = (y_val > q_predictions['p90']).mean()
in_80_interval = ((y_val >= q_predictions['p10']) & (y_val <= q_predictions['p90'])).mean()

print(f'\n=== Prediction Interval Coverage ===')
print(f'  Below p10: {below_p10:.1%} (target: 10%)')
print(f'  Above p90: {above_p90:.1%} (target: 10%)')
print(f'  Within p10-p90: {in_80_interval:.1%} (target: 80%)')

## 7. Bias-Variance Analysis by Segment

Disaggregate bias analysis reveals whether the model systematically over/under-forecasts specific product categories — critical for warehouse stock policy.

In [ ]:
# 7.1 Bias by category and ABC class
val_df_eval = val_df.copy()
val_df_eval['pred'] = preds[best_name]
val_df_eval['residual'] = val_df_eval['demand_units'] - val_df_eval['pred']
val_df_eval['bias'] = val_df_eval['pred'] - val_df_eval['demand_units']  # positive = over-forecast
val_df_eval['abs_error'] = np.abs(val_df_eval['residual'])
val_df_eval['pct_error'] = val_df_eval['abs_error'] / (val_df_eval['demand_units'] + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bias by category
cat_bias = val_df_eval.groupby('fg_category')['bias'].mean().sort_values()
colors = ['green' if b < 0 else 'red' for b in cat_bias.values]
axes[0].barh(cat_bias.index, cat_bias.values, color=colors)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_title(f'Mean Bias by Category — {best_name}')
axes[0].set_xlabel('Bias (positive = over-forecast)')

# Bias by demand level (low/medium/high volume)
val_df_eval['demand_tier'] = pd.qcut(val_df_eval['demand_units'], q=3,
                                      labels=['Low', 'Medium', 'High'])
tier_bias = val_df_eval.groupby('demand_tier')['bias'].agg(['mean', 'std'])
axes[1].bar(tier_bias.index, tier_bias['mean'],
            yerr=tier_bias['std']/np.sqrt(len(val_df_eval)),
            capsize=5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Bias by Demand Tier')
axes[1].set_ylabel('Mean Bias')

# WAPE by category
cat_wape = val_df_eval.groupby('fg_category').apply(
    lambda g: np.sum(g['abs_error']) / max(np.sum(g['demand_units']), 1)
).sort_values(ascending=False)
axes[2].barh(cat_wape.index, cat_wape.values)
axes[2].axvline(0.10, color='green', linestyle='--', label='Target (10%)')
axes[2].set_title('WAPE by Category')
axes[2].set_xlabel('WAPE')
axes[2].legend()

plt.suptitle('Bias-Variance Analysis by Segment', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Under-forecast analysis (warehouse risk)
underforecast_rate = (val_df_eval['bias'] < 0).mean()
print(f'\nUnder-forecast rate: {underforecast_rate:.1%}')
print(f'Over-forecast rate: {1-underforecast_rate:.1%}')
print('Note: In warehouse operations, under-forecasting leads to stockouts (worse than over-forecasting)')

## 8. Evaluation Summary

### Key Findings

In [ ]:
# 8.1 Comprehensive evaluation table
eval_summary = []
for name, pred in preds.items():
    metrics = {
        'Model': name,
        'R²': r2_score(y_val, pred),
        'RMSE': np.sqrt(mean_squared_error(y_val, pred)),
        'MAE': mean_absolute_error(y_val, pred),
        'WAPE': np.sum(np.abs(y_val - pred)) / max(np.sum(np.abs(y_val)), 1),
        'Mean MASE': np.mean(mase_per_sku.get(name, [np.nan])),
        'Bias': np.mean(pred - y_val),
    }
    eval_summary.append(metrics)

eval_df = pd.DataFrame(eval_summary)
eval_df = eval_df.sort_values('WAPE')

print('='*80)
print('  COMPREHENSIVE MODEL EVALUATION SUMMARY')
print('='*80)
print(eval_df.to_string(index=False, float_format='%.4f'))

print(f'\n--- Diagnostic Tests ---')
print(f'Shapiro-Wilk (residual normality): W={shapiro_stat:.4f}, p={shapiro_p:.6f}')
print(f'Ljung-Box (residual autocorrelation): p={lb_result["lb_pvalue"].iloc[-1]:.6f}')
print(f'80% PI Coverage: {in_80_interval:.1%}')
print(f'\n--- Diebold-Mariano Tests ---')
for r in dm_results:
    print(f'  {r["model"]} vs Naive: DM={r["dm_stat"]:.3f}, p={r["p_value"]:.6f} -> {r["conclusion"]}')

# Save evaluation
eval_df.to_csv(ENG_DIR / 'model_evaluation_summary.csv', index=False)
print(f'\nSaved to: {ENG_DIR / "model_evaluation_summary.csv"}')
print(f'\nNext: Notebook 05 — ML vs Statistical Model Comparison (M5 Real Data vs Synthetic)')